# RAG Generation with Oracle DB + Ollama (llama3.2)

This notebook picks up after retrieval (`RAG_Query_OracleDB.ipynb`).
It:
1. Connects to Oracle DB and retrieves relevant chunks for a query
2. Builds a prompt with the retrieved context
3. Uses the local `llama3.2` model via Ollama to generate an answer

**Prerequisites:** Ollama must be running locally (`ollama serve`) with `llama3.2` pulled.

## 1. Install dependencies (run once)

In [1]:
# Run this cell once if packages are not already installed
# !pip install oracledb langchain-oracledb langchain-ollama langchain-core

## 2. Connect to Oracle DB

In [2]:
import os
import oracledb

connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASS"],
    dsn=os.environ["ORACLE_TLS_DNS"]
)

print("Connected to Oracle DB")

Connected to Oracle DB


## 3. Set up the embedding model and vector store (retrieval)

Using the same `nomic-embed-text` embeddings via Ollama as in the query notebook.

In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_oracledb.vectorstores import DistanceStrategy

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Change TABLE_NAME to switch between document collections
TABLE_NAME = "SOLUTION_ARCHITECT_VECTORS"  # or "DOCUMENT_VECTORS"

vector_store = OracleVS(
    client=connection,
    table_name=TABLE_NAME,
    embedding_function=embedding_model,
    distance_strategy=DistanceStrategy.COSINE
)

print(f"Vector store ready — table: {TABLE_NAME}")

/Users/zainabfirdaus/git/airflow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector store ready — table: SOLUTION_ARCHITECT_VECTORS


## 4. Load the Ollama LLM (llama3.2)

`langchain-ollama` talks to your local Ollama server — no model loading into Python memory, no GPU management needed.

In [4]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3.2",
    temperature=0.1,       # low temperature for factual RAG answers
    num_predict=256,       # max tokens to generate
)

# Quick sanity check — confirms Ollama is reachable
print(llm.invoke("Reply with just: OK"))

OK


## 5. Define the RAG pipeline

Steps:
1. Retrieve top-k chunks from Oracle vector store
2. Build a grounded prompt with the retrieved context
3. Generate an answer with `llama3.2` via Ollama

In [5]:
def retrieve_chunks(query: str, k: int = 3) -> list:
    """Retrieve top-k relevant chunks from the Oracle vector store."""
    return vector_store.similarity_search(query, k=k)


def build_prompt(query: str, chunks: list) -> str:
    """Build a RAG prompt with the retrieved context."""
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        source = chunk.metadata.get("source", "unknown")
        context_parts.append(f"[{i}] (source: {source})\n{chunk.page_content.strip()}")

    context = "\n\n".join(context_parts)

    prompt = (
        "You are a helpful assistant. Use only the context below to answer the question. "
        "If the answer is not in the context, say 'I don't know based on the provided context.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )
    return prompt


def rag_answer(query: str, k: int = 3) -> tuple:
    """End-to-end RAG: retrieve → prompt → generate."""
    chunks = retrieve_chunks(query, k=k)
    prompt = build_prompt(query, chunks)
    answer = llm.invoke(prompt)
    return answer, chunks, prompt


print("RAG pipeline functions defined")

RAG pipeline functions defined


## 6. Run a query end-to-end

In [6]:
query = "What is DRM and how does it work in a cloud architecture?"

answer, chunks, prompt = rag_answer(query, k=3)

# ── Retrieved chunks ───────────────────────────────────────────────────────
print("=" * 60)
print("RETRIEVED CHUNKS")
print("=" * 60)
for i, chunk in enumerate(chunks, 1):
    print(f"\n[{i}] source: {chunk.metadata.get('source')}")
    print(chunk.page_content.strip())

# ── Generated answer ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("GENERATED ANSWER")
print("=" * 60)
print(answer)

RETRIEVED CHUNKS

[1] source: media_security_cloud.txt
(DRM) securely controls the keys required to unlock it. DRM interfaces directly with a device's built-in operating system or browser Content Decryption Module (CDM) so keys are never exposed in plaintext memory.A proper cloud architecture requires a Multi-DRM Setup to target different ecosystems natively:DRM EcosystemNative EnvironmentsMain StrengthsGoogle WidevineAndroid, Chrome, Firefox, ChromecastOpen-source foundation; flexible security levels (L1 hardware vs L3 software).Apple FairPlayiOS,

[2] source: media_security_cloud.txt
invisible, unique mathematical signature into the video frames during cloud packaging. If a leaked stream surfaces online, scanning companies can analyze the clip, extract the invisible watermark, and trace it back to the exact subscriber account that initiated the leak.💡 Architecture Tip: For optimal implementation, host your decryption keys and DRM certificates in secure, dedicated cloud storage like a

## 7. Streaming output (optional)

Stream tokens as they are generated instead of waiting for the full response.

In [ ]:
stream_query = "What is DRM and how does it work in a cloud architecture?"

chunks_retrieved = retrieve_chunks(stream_query, k=3)
stream_prompt = build_prompt(stream_query, chunks_retrieved)

print(f"Q: {stream_query}\n")
print("A: ", end="", flush=True)

for token in llm.stream(stream_prompt):
    print(token, end="", flush=True)

print()  # newline at the end

## 8. Inspect the full prompt (debug)

In [ ]:
print(prompt)

## 9. Try a custom query

In [ ]:
# Edit and re-run with any query
custom_query = "What is Oracle vector search?"

answer, _, _ = rag_answer(custom_query, k=3)
print(f"Q: {custom_query}\n")
print(f"A: {answer}")

## 10. Batch evaluation

In [ ]:
test_queries = [
    "What is HNSW?",
    "How does Widevine DRM work?",
    "What is cosine similarity in vector search?",
]

for q in test_queries:
    ans, _, _ = rag_answer(q, k=2)
    print(f"Q: {q}")
    print(f"A: {ans}")
    print("-" * 50)